# Stage 1 PINN: multi-seed experiment on Google Colab
Run the cells **in order**. Everything is stored in your Google Drive (folder `pinn_stage1`), so if Colab disconnects, nothing is lost: reconnect, run cells 1–2 again, then rerun the cell you were on.

**Before starting:** create the folder `pinn_stage1` in **My Drive** and upload these four files into it: `pinn_stage1_jax.py`, `run_all.sh`, `summarize_results.py` (and optionally this notebook).

## 1. Connect Google Drive
A pop-up asks you to allow access to your Drive; accept it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Go to the project folder and check the files are there

In [ ]:
%cd /content/drive/MyDrive/pinn_stage1
!ls -l

## 3. Check JAX and the hardware
Shows whether this runtime has a GPU (`CudaDevice`) or only a CPU (`CpuDevice`). To switch: *Runtime → Change runtime type*, then run cells 1–3 again.

In [ ]:
import jax, scipy, numpy
print('JAX', jax.__version__, '| SciPy', scipy.__version__, '| NumPy', numpy.__version__)
print(jax.devices())
!nproc

## 4. Self-test (must pass before anything else)
Both `U` values on the first line must agree to all 12 digits, and every residual must be tiny (about 1e-12 or smaller). If not, stop and send the output.

In [ ]:
!python pinn_stage1_jax.py --mode selftest
!python pinn_stage1_jax.py --mode selftest --branch held

## 5. Trial run: how long does one run take here?
One baseline run (2000 interior points, 2100 L-BFGS iterations). The training line ends with `checkpoint ... saved (N s)`: **N is the time for one run.** A healthy `rel.err` lies roughly between 1e-5 and 3e-3.

Optional: run this cell once on a CPU runtime and once on a GPU runtime and keep whichever is faster for step 6.

In [ ]:
!rm -f ckpt_trial.npz
!python pinn_stage1_jax.py --mode lbfgs --iters 2100 --n_int 2000 --seed 0 --tag trial
!python pinn_stage1_jax.py --mode eval --n_int 2000 --seed 0 --tag trial | head -20

In [ ]:
# remove the trial files so they do not mix with the experiment
!rm -f ckpt_trial.npz results_trial.json fields_trial.npz

## 6. Full experiment (20 runs)
Configurations A, B, C, H with seeds 0–4 (see the header of `run_all.sh`). Expected total time ≈ 20 × the time from step 5.

**If Colab disconnects:** reconnect, run cells 1–2, then run this cell again. Finished runs are skipped; an interrupted run restarts from scratch so every run has the same budget.

Keep this browser tab open while it runs; the free tier disconnects idle sessions.

In [ ]:
!bash run_all.sh

## 7. Summary
Prints the median and range over seeds for each configuration and writes `summary.csv`.

In [ ]:
!python summarize_results.py

In [ ]:
# run times per run (for the cost comparison)
!cat logs/timing.csv

## 8. Send back
Copy the output of the two cells above into the chat. All files (results, fields, logs) stay in `My Drive/pinn_stage1`.